# Problem C — Pre-Flight Risk Scoring: Visualizations

Replicates and extends all charts from the Streamlit **Problem C** page.

| Section | Charts |
|---------|--------|
| Performance | ROC curve, PR curve, lift chart |
| Risk Distribution | Predicted probability histogram, risk level breakdown |
| Sensitivity | Weather parameter sensitivity curves |
| Data EDA | Feature distributions, weather vs incident rate heatmap |

**Pre-req:** Run `scripts/run_phase3.py` to generate `models/preflight_lgbm_calibrated.joblib`.

In [1]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

Working directory: e:\OSFDA


In [2]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

MODELS    = Path("models")
PROCESSED = Path("data/processed")

_PREFLIGHT_DEFAULT_FEATURES = [
    "month_sin", "month_cos", "dow_sin", "dow_cos", "hour",
    "temp", "rhum", "prcp", "wspd", "pres",
    "airport_risk_rate", "carrier_risk_rate", "route_risk_rate", "DISTANCE",
]

print("Libraries loaded.")

Libraries loaded.


## 1. Load Model & Test Data

In [3]:
import sys
import joblib
import src.models.preflight  # register custom classes

artifact = joblib.load(MODELS / "preflight_lgbm_calibrated.joblib")
model    = artifact["model"]   if isinstance(artifact, dict) and "model"    in artifact else artifact
features = artifact["features"] if isinstance(artifact, dict) and "features" in artifact else _PREFLIGHT_DEFAULT_FEATURES

df = pd.read_parquet(PROCESSED / "problem_c" / "preflight_features_final.parquet")

# Resolve year
if "year" not in df.columns:
    for date_col in ("FL_DATE", "fl_date", "date"):
        if date_col in df.columns:
            df["year"] = pd.to_datetime(df[date_col], errors="coerce").dt.year
            break
    else:
        df["year"] = 2020

test_df  = df[df["year"] == 2020].copy()
features = [f for f in features if f in test_df.columns]
X_test   = test_df[features]
y_test   = test_df["incident"]

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(f"Test flights:   {len(y_test):,}")
print(f"Incident rate:  {y_test.mean():.4%}")
print(f"Features used:  {features}")

Test flights:   153,018
Incident rate:  4.7615%
Features used:  ['month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'hour', 'temp', 'rhum', 'prcp', 'wspd', 'pres', 'airport_risk_rate', 'carrier_risk_rate', 'route_risk_rate', 'DISTANCE']


## 2. Performance Metrics

In [4]:
roc_auc = roc_auc_score(y_test, y_prob)
ap      = average_precision_score(y_test, y_prob)

print(f"ROC-AUC:         {roc_auc:.4f}  (target 0.70–0.80)")
print(f"Avg Precision:   {ap:.4f}  (PR-AUC, primary metric for imbalanced data)")
print(f"Positive Rate:   {y_test.mean():.4%}")

ROC-AUC:         0.5975  (target 0.70–0.80)
Avg Precision:   0.0587  (PR-AUC, primary metric for imbalanced data)
Positive Rate:   4.7615%


## 3. ROC Curve

In [5]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    name=f"LightGBM (AUC = {roc_auc:.3f})",
    line=dict(color="#e74c3c", width=2),
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    name="Random (AUC = 0.5)",
    line=dict(color="gray", dash="dash"),
))
fig.update_layout(
    title="ROC Curve — Pre-Flight Risk Model (Test Set 2020)",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=450,
)
fig.show()

## 4. Precision-Recall Curve

In [6]:
prec, rec, _ = precision_recall_curve(y_test, y_prob)
baseline     = y_test.mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rec, y=prec,
    name=f"LightGBM (AP = {ap:.3f})",
    line=dict(color="#3498db", width=2),
))
fig.add_hline(
    y=baseline,
    line_dash="dash", line_color="gray",
    annotation_text=f"Baseline ({baseline:.4%})",
)
fig.update_layout(
    title="Precision-Recall Curve — Pre-Flight Risk Model",
    xaxis_title="Recall",
    yaxis_title="Precision",
    height=450,
)
fig.show()

## 5. Cumulative Lift Chart

In [7]:
n          = len(y_test)
sorted_idx = np.argsort(-y_prob)
cumpos     = np.cumsum(y_test.values[sorted_idx])
total_pos  = y_test.sum()
lift       = (cumpos / total_pos) / (np.arange(1, n + 1) / n)
sample_pct = np.linspace(0, 100, n)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sample_pct, y=lift,
    name="Model Lift",
    line=dict(color="#27ae60", width=2),
))
fig.add_hline(y=1.0, line_dash="dash", line_color="gray", annotation_text="No lift (random)")
# Annotate lift at 5% and 10%
for pct in [5, 10, 20]:
    idx = int(pct / 100 * n)
    if idx < len(lift):
        fig.add_annotation(
            x=pct, y=lift[idx],
            text=f"x{lift[idx]:.1f}",
            showarrow=True, arrowhead=1,
        )
fig.update_layout(
    title="Cumulative Lift — Model vs. Random Screening",
    xaxis_title="% of Flights Screened (ranked by risk score)",
    yaxis_title="Cumulative Lift",
    height=400,
)
fig.show()

## 6. Predicted Risk Probability Distribution

In [8]:
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=y_prob[y_test == 0] * 100,
    name="Non-incident",
    opacity=0.6,
    marker_color="steelblue",
    nbinsx=80,
))
fig.add_trace(go.Histogram(
    x=y_prob[y_test == 1] * 100,
    name="Incident",
    opacity=0.6,
    marker_color="#e74c3c",
    nbinsx=80,
))
fig.update_layout(
    barmode="overlay",
    title="Predicted Risk Score Distribution (Incident vs Non-Incident Flights)",
    xaxis_title="Risk Score (%)",
    yaxis_title="Flight Count",
    height=380,
)
fig.show()

## 7. Risk Level Breakdown (LOW / MEDIUM / HIGH)

In [9]:
def risk_level(p):
    p_pct = p * 100
    if p_pct < 0.03:  return "LOW"
    if p_pct < 0.08:  return "MEDIUM"
    return "HIGH"

risk_df = pd.DataFrame({"prob": y_prob, "incident": y_test.values})
risk_df["level"] = risk_df["prob"].apply(risk_level)

level_stats = (
    risk_df.groupby("level")
    .agg(flights=("prob", "count"), incidents=("incident", "sum"))
    .assign(incident_rate=lambda d: d["incidents"] / d["flights"])
    .reindex(["LOW", "MEDIUM", "HIGH"])
)
print(level_stats.to_string())

fig = px.bar(
    level_stats.reset_index(),
    x="level",
    y="flights",
    color="level",
    color_discrete_map={"LOW": "#2ecc71", "MEDIUM": "#f39c12", "HIGH": "#e74c3c"},
    title="Flight Count by Risk Level",
    labels={"level": "Risk Level", "flights": "Flight Count"},
    category_orders={"level": ["LOW", "MEDIUM", "HIGH"]},
)
fig.update_layout(height=350, showlegend=False)
fig.show()

         flights  incidents  incident_rate
level                                     
LOW          NaN        NaN            NaN
MEDIUM       NaN        NaN            NaN
HIGH    153018.0     7286.0       0.047615


## 8. Weather Parameter Sensitivity Curves

Holds all features at median values, varies one weather parameter at a time.

In [10]:
medians = X_test.median().to_dict()

def predict_sweep(param, values):
    rows = []
    for v in values:
        row = {**medians, param: v}
        rows.append(row)
    df_sweep = pd.DataFrame(rows)[features]
    return model.predict_proba(df_sweep)[:, 1] * 100

weather_params = {
    "prcp":  (np.linspace(0, 2.0, 40),  "Precipitation (in)"),
    "wspd":  (np.linspace(0, 50, 40),   "Wind Speed (mph)"),
    "temp":  (np.linspace(-20, 110, 40), "Temperature (°F)"),
    "rhum":  (np.linspace(0, 100, 40),  "Humidity (%)"),
}

fig = go.Figure()
palette = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]

for (param, (values, xlabel)), color in zip(weather_params.items(), palette):
    if param not in features:
        continue
    risks = predict_sweep(param, values)
    fig.add_trace(go.Scatter(
        x=values, y=risks,
        name=xlabel,
        line=dict(color=color, width=2),
    ))

fig.update_layout(
    title="Risk Sensitivity to Weather Parameters (all others held at median)",
    xaxis_title="Parameter Value",
    yaxis_title="Predicted Risk (%)",
    height=430,
    hovermode="x unified",
)
fig.show()

## 9. Feature Importance (LightGBM Gain)

In [11]:
try:
    lgbm_booster = model.named_steps["clf"] if hasattr(model, "named_steps") else model
    imp     = lgbm_booster.booster_.feature_importance(importance_type="gain")
    fn      = lgbm_booster.booster_.feature_name()
except Exception:
    # Try direct booster access (CalibratedClassifierCV wraps the estimator)
    try:
        base_est = model.calibrated_classifiers_[0].estimator
        imp      = base_est.booster_.feature_importance(importance_type="gain")
        fn       = base_est.booster_.feature_name()
    except Exception:
        imp = None
        print("Could not extract feature importances directly.")

if imp is not None:
    top_idx = np.argsort(imp)[::-1]
    fig = px.bar(
        x=imp[top_idx],
        y=[fn[i] for i in top_idx],
        orientation="h",
        title="Feature Importance (Gain) — Pre-flight LightGBM",
        labels={"x": "Importance (gain)", "y": "Feature"},
        color=imp[top_idx],
        color_continuous_scale="Blues",
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(height=400, showlegend=False, margin=dict(l=200))
    fig.show()

## 10. Weather Heatmap: Average Risk by Precipitation × Wind Speed Bins

In [12]:
if "prcp" in features and "wspd" in features:
    prcp_bins = [0, 0.1, 0.25, 0.5, 1.0, 2.0]
    wspd_bins = [0,   5,   10,  20,  35,  50]

    risk_df2 = X_test.copy()
    risk_df2["risk"]      = y_prob
    risk_df2["prcp_bin"]  = pd.cut(risk_df2["prcp"], bins=prcp_bins, include_lowest=True)
    risk_df2["wspd_bin"]  = pd.cut(risk_df2["wspd"], bins=wspd_bins, include_lowest=True)

    pivot = (
        risk_df2.groupby(["prcp_bin", "wspd_bin"])["risk"]
        .mean()
        .mul(100)
        .unstack("wspd_bin")
    )

    fig = px.imshow(
        pivot.values,
        x=[str(c) for c in pivot.columns],
        y=[str(r) for r in pivot.index],
        color_continuous_scale="RdYlGn_r",
        title="Average Predicted Risk (%) by Precipitation × Wind Speed",
        labels={"x": "Wind Speed bin (mph)", "y": "Precipitation bin (in)", "color": "Risk (%)"},
        text_auto=".3f",
    )
    fig.update_layout(height=400)
    fig.show()
else:
    print("prcp or wspd not in features — skipping heatmap.")

C:\Users\yashn\AppData\Local\Temp\ipykernel_412\4202163459.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  risk_df2.groupby(["prcp_bin", "wspd_bin"])["risk"]
